# IOAI — 2024 Mock Competition Real Or Fake Image — ⭐모범답안 (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/jaredliw/ioai-tsp-2025"
CLONE = "ioai-tsp-2025"
SUBDIR = "noai-china-2024/real-or-fake-image"
WORKDIR = "noai-china-2024/real-or-fake-image"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Real or Fake Image Recognition

> **NOAI China 2024 — Question 2** (APOAI 2025 Mock, Q2).

Tell **real** CIFAR-10 photos (`train_v1/cifar`, label **0**) from **fake** diffusion-generated
images (`train_v1/uvit`, label **1**) with a small **CNN**.

**Constraints (enforced by the official grader):** at least 2 `nn.Conv2d` + 2 `nn.MaxPool2d`,
at most 2 `nn.Linear`, build directly (no `nn.Sequential`), activations from ReLU/Sigmoid/Tanh/...
Official score `= (1/(num_linear+num_conv+1) + accuracy) * 3/4`.

Official A/B test sets are hidden, so we hold out 20% (`index % 5 == 0` within each class)
as validation. The **Submit** button re-runs your saved model on the same held-out split.

In [ ]:
import os, random, zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## Dataset with a deterministic held-out split

`cifar`→0, `uvit`→1. Within each class, sorted-by-name, `index % 5 == 0` is validation.

In [ ]:
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),   # the grader uses the same normalization
])

class ImgDS(Dataset):
    def __init__(self, root, split):
        self.items = []
        for label, sub in enumerate(['cifar', 'uvit']):   # cifar=0(real), uvit=1(fake)
            files = sorted(os.listdir(os.path.join(root, sub)))
            for i, fn in enumerate(files):
                is_val = (i % 10 == 0)
                if (split == 'val') == is_val:
                    self.items.append((os.path.join(root, sub, fn), label))
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        return transform(Image.open(p).convert('RGB')), label

ds_tr = ImgDS('./train_v1', 'train')
ds_va = ImgDS('./train_v1', 'val')
dl_tr = DataLoader(ds_tr, batch_size=64, shuffle=True)
dl_va = DataLoader(ds_va, batch_size=128)
len(ds_tr), len(ds_va)

## Define the CNN (2 Conv + 2 MaxPool + 2 Linear, no Sequential) and train

In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(8, 10, 5)
        self.fc1 = nn.Linear(10 * 5 * 5, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 10 * 5 * 5)
        x = self.fc1(x)
        return self.sigmoid(x)

In [ ]:
@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device).float()
        preds = (model(x).squeeze() >= 0.5)
        correct += (preds == y).sum().item(); total += y.size(0)
    return correct / total

model = MyModel().to(device)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=30)
best=model
for epoch in range(30):
    model.train()
    for x, y in dl_tr:
        x, y = x.to(device), y.to(device).float()
        loss = criterion(model(x).squeeze(), y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    print(f'epoch {epoch+1}  val_acc {accuracy(model, dl_va):.4f}')
    scheduler.step()

In [ ]:
val_acc = accuracy(model, dl_va)
print(f'Held-out validation accuracy: {val_acc:.4f}')

## Save submission (`submission.zip`)

In [ ]:
torch.save(model.state_dict(), 'submission_dic.pth')
model_code = '''import torch
import torch.nn as nn
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(8, 10, 5)
        self.fc1 = nn.Linear(10 * 5 * 5, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(-1, 10 * 5 * 5)
        x = self.fc1(x)
        return self.sigmoid(x)
'''
with open('submission_model.py', 'w') as f:
    f.write(model_code)
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('submission_model.py'); z.write('submission_dic.pth')
print('wrote submission.zip')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)